In [45]:
import pandas as pd
import os

# ============================
# Paths
# ============================
super_path = "/Users/judycheng/Desktop/supercharger in washington state.xls"
output_path = "/Users/judycheng/Desktop/supercharger_by_county_summary.xlsx"

# ============================
# Read file
# ============================
df = pd.read_excel(super_path)

# ============================
# Filter Washington
# ============================
df_wa = df[df["State"] == "Washington"]

# ============================
# Count all charge *points*  
# (1 row = 1 charger point)
# ============================
counts = (
    df_wa.groupby("County")["County"]
    .count()
    .reset_index(name="Supercharger_Count")
)

# ============================
# Clean "County" suffix
# ============================
counts["County"] = counts["County"].str.replace(" County", "", regex=False)

# Sort descending
counts = counts.sort_values("Supercharger_Count", ascending=False)

# ============================
# Export to Excel
# ============================
counts.to_excel(output_path, index=False)

print("Done! File exported to:", output_path)


Done! File exported to: /Users/judycheng/Desktop/supercharger_by_county_summary.xlsx


In [42]:
import pandas as pd

# =========================
# Paths
# =========================
residents_path = "/Users/judycheng/Desktop/Population 2024 age 25 to 59.xlsx"
super_summary_path = "/Users/judycheng/Desktop/supercharger_by_county_summary.xlsx"
ev_path = "/Users/judycheng/Desktop/coordinates_output.xlsm"

output_forecast_path = (
    "/Users/judycheng/Desktop/wa_county_ev_forecast_baseline_2024_to_2050.xlsx"
)

# =========================
# 1) Read Residents (Population) file
# =========================
residents_df = pd.read_excel(residents_path)

# Clean County names
residents_df["County"] = (
    residents_df["County"].astype(str)
    .str.replace(" County", "", regex=False)
    .str.strip()
)

# ❗ Explicitly use the "total" column as Pop_2024
if "total" not in residents_df.columns:
    raise ValueError("The residents file does not contain a 'total' column.")

pop_df = residents_df[["County", "total"]].rename(columns={"total": "Pop_2024"})

# =========================
# 2) Read Supercharger summary (points per county)
# =========================
super_df = pd.read_excel(super_summary_path)
super_df["County"] = super_df["County"].astype(str).str.strip()

super_df = super_df.rename(columns={"Supercharger_Count": "Superchargers_2024"})
super_df = super_df[["County", "Superchargers_2024"]]

# =========================
# 3) Read EV registration file (VIN count)
# =========================
ev_df = pd.read_excel(ev_path)
ev_df["County"] = (
    ev_df["County"].astype(str)
    .str.replace(" County", "", regex=False)
    .str.strip()
)

ev_counts = (
    ev_df.groupby("County")["County"]
    .count()
    .reset_index(name="EVs_2024")
)

# =========================
# 4) Build 2024 baseline
# =========================
base = pop_df.merge(super_df, on="County", how="left")
base = base.merge(ev_counts, on="County", how="left")

base["Superchargers_2024"] = base["Superchargers_2024"].fillna(0).astype(int)
base["EVs_2024"] = base["EVs_2024"].fillna(0).astype(int)

base["Adoption_2024"] = (
    base["EVs_2024"] / base["Pop_2024"]
).fillna(0)

# Order columns
base = base[["County", "Pop_2024", "Superchargers_2024", "EVs_2024", "Adoption_2024"]]

# =========================
# 5) Add 2025–2050 empty forecast columns
# =========================
for year in range(2025, 2051):
    base[f"Superchargers_{year}"] = pd.NA
    base[f"EVs_{year}"] = pd.NA
    base[f"Adoption_{year}"] = pd.NA

# Sort alphabetically
base = base.sort_values("County").reset_index(drop=True)

# =========================
# 6) Export to Excel
# =========================
base.to_excel(output_forecast_path, index=False)

print("✅ Forecast baseline file created:")
print(output_forecast_path)


✅ Forecast baseline file created:
/Users/judycheng/Desktop/wa_county_ev_forecast_baseline_2024_to_2050.xlsx


In [16]:
import pandas as pd

# ========================================================
# FILE PATHS
# ========================================================
base_path = "/Users/judycheng/Desktop/wa_county_ev_forecast_baseline_2024_to_2050.xlsx"

king_path   = "/Users/judycheng/Desktop/king_county_ev_projection_mc_monotonic.xlsx"
pierce_path = "/Users/judycheng/Desktop/pierce_county_ev_projection_mc_monotonic.xlsx"
kitsap_path = "/Users/judycheng/Desktop/kitsap_county_ev_projection_mc_monotonic.xlsx"
chelan_path = "/Users/judycheng/Desktop/chelan_county_ev_projection_mc_monotonic.xlsx"

output_path = "/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc.xlsx"

YEARS = list(range(2025, 2051))


# ========================================================
# 1. LOAD BASELINE FILE
# ========================================================
df = pd.read_excel(base_path)
df["County"] = df["County"].astype(str).str.strip()


# ========================================================
# 2. LOAD MC TEMPLATE FUNCTION
# ========================================================
def load_mc(path):
    mc = pd.read_excel(path, sheet_name="Forecast")
    mc = mc.rename(columns={mc.columns[0]: "Year"})
    mc = mc.set_index("Year")

    needed = ["Forecast_Chargers", "Forecast_EVs_P50", "Forecast_Adoption_P50"]
    for c in needed:
        if c not in mc.columns:
            raise ValueError(f"Missing column {c} in template: {path}")

    return mc[needed]


# Load all 4 MC templates
king_mc   = load_mc(king_path)
pierce_mc = load_mc(pierce_path)
kitsap_mc = load_mc(kitsap_path)
chelan_mc = load_mc(chelan_path)


# ========================================================
# 3. SELECT TEMPLATE BASED ON POPULATION
# ========================================================
def select_template(pop):
    if pop > 1_000_000:
        return king_mc
    elif pop > 130_000:
        return pierce_mc
    elif pop > 34_000:
        return kitsap_mc
    else:
        return chelan_mc


# ========================================================
# 4. FILL IN FORECAST VALUES FOR 2025–2050
# ========================================================
for idx, row in df.iterrows():
    pop = row["Pop_2024"]
    template = select_template(pop)

    for year in YEARS:

        # Read adoption and chargers from MC template
        adoption = template.loc[year, "Forecast_Adoption_P50"]
        chargers = template.loc[year, "Forecast_Chargers"]

        # EVs = Adoption × Pop_2024
        evs = adoption * pop

        df.at[idx, f"Adoption_{year}"]      = adoption
        df.at[idx, f"Superchargers_{year}"] = chargers
        df.at[idx, f"EVs_{year}"]           = evs


# Optional: convert numeric
for year in YEARS:
    df[f"Superchargers_{year}"] = pd.to_numeric(df[f"Superchargers_{year}"], errors="coerce")
    df[f"EVs_{year}"]           = pd.to_numeric(df[f"EVs_{year}"], errors="coerce")
    df[f"Adoption_{year}"]      = pd.to_numeric(df[f"Adoption_{year}"], errors="coerce")

# ========================================================
# 4.5 ADD STATEWIDE TOTAL ROW
# ========================================================

# Total population (used for weighted adoption)
total_pop = df["Pop_2024"].sum()

total_row = {
    "County": "TOTAL",
    "Pop_2024": total_pop
}

for year in YEARS:

    # --- Sum EVs and Superchargers ---
    total_evs = df[f"EVs_{year}"].sum()
    total_sc  = df[f"Superchargers_{year}"].sum()

    # --- Weighted adoption ---
    weighted_adoption = (
        (df[f"Adoption_{year}"] * df["Pop_2024"]).sum()
        / total_pop
    )

    total_row[f"EVs_{year}"]           = total_evs
    total_row[f"Superchargers_{year}"] = total_sc
    total_row[f"Adoption_{year}"]      = weighted_adoption

# Append TOTAL row
df = pd.concat([df, pd.DataFrame([total_row])], ignore_index=True)


# ========================================================
# 5. EXPORT FINAL FILE
# ========================================================
df.to_excel(output_path, index=False)

print("✅ Monte Carlo forecast (2025–2050) successfully filled.")
print("📄 Output saved to:")
print(output_path)


✅ Monte Carlo forecast (2025–2050) successfully filled.
📄 Output saved to:
/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc.xlsx


In [52]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# ======================================================
# PATHS
# ======================================================
input_path = "/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc.xlsx"

desktop = os.path.join(os.path.expanduser("~"), "Desktop")
output_folder = os.path.join(desktop, "county_charts_final")
os.makedirs(output_folder, exist_ok=True)

# ======================================================
# LOAD DATA
# ======================================================
df = pd.read_excel(input_path)

YEARS = np.array(range(2024, 2051))
IDX   = np.arange(len(YEARS))   # even spacing

# ======================================================
# FIX TOTAL ROW — FILL 2024 VALUES
# ======================================================
mask_total = df["County"].astype(str).str.strip() == "TOTAL"

if mask_total.any():

    non_total = df.loc[~mask_total]

    # --- Superchargers & EVs: SUM ---
    df.loc[mask_total, "Superchargers_2024"] = non_total["Superchargers_2024"].sum()
    df.loc[mask_total, "EVs_2024"]           = non_total["EVs_2024"].sum()

    # --- Adoption: POPULATION-WEIGHTED ---
    weighted_adoption_2024 = (
        (non_total["Adoption_2024"] * non_total["Pop_2024"]).sum()
        / non_total["Pop_2024"].sum()
    )

    df.loc[mask_total, "Adoption_2024"] = weighted_adoption_2024

# ======================================================
# PLOT EACH COUNTY + TOTAL
# ======================================================
for _, row in df.iterrows():

    county = str(row["County"]).strip()

    superchargers = np.array(
        [row[f"Superchargers_{y}"] for y in YEARS], dtype=float
    )
    adoption = np.array(
        [row[f"Adoption_{y}"] for y in YEARS], dtype=float
    )

    # --------------------------------------------------
    # FIGURE
    # --------------------------------------------------
    fig, ax = plt.subplots(figsize=(11, 7))

    # --------------------------------------------------
    # MAIN PLOT — Y: SUPERCHARGERS
    # --------------------------------------------------
    ax.plot(IDX, superchargers, linewidth=2)
    ax.set_ylabel("Superchargers")

    # --------------------------------------------------
    # BOTTOM X-AXIS — ADOPTION (VERTICAL)
    # --------------------------------------------------
    ax.set_xticks(IDX)
    ax.set_xticklabels(
        [f"{a*100:.0f}%" for a in adoption],
        rotation=90,
        fontsize=8
    )
    ax.set_xlabel("EV Adoption Rate", labelpad=30)

    # --------------------------------------------------
    # TOP X-AXIS — YEAR (VERTICAL)
    # --------------------------------------------------
    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(IDX)
    ax_top.set_xticklabels(YEARS, rotation=90, fontsize=8)
    ax_top.set_xlabel("Year", labelpad=30)

    # --------------------------------------------------
    # TITLE
    # --------------------------------------------------
    ax.set_title(county, fontsize=12)

    # --------------------------------------------------
    # SAVE (COUNTY NAME ONLY)
    # --------------------------------------------------
    safe_name = county.replace(" ", "_").replace("/", "_")
    out_path = os.path.join(output_folder, f"{safe_name}.png")

    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()

print("✅ TOTAL row fixed and charts generated successfully")
print("📁 Output folder:", output_folder)


✅ TOTAL row fixed and charts generated successfully
📁 Output folder: /Users/judycheng/Desktop/county_charts_final


In [53]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# ======================================================
# PATHS
# ======================================================
input_path  = "/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc.xlsx"
output_path = "/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc_TOTAL_FIXED.xlsx"

desktop = os.path.join(os.path.expanduser("~"), "Desktop")
output_folder = os.path.join(desktop, "county_charts_scatter")
os.makedirs(output_folder, exist_ok=True)

# ======================================================
# LOAD DATA
# ======================================================
df = pd.read_excel(input_path)
df["County"] = df["County"].astype(str).str.strip()

# ======================================================
# DEFINE TIME VARIABLES
# ======================================================
YEARS = np.arange(2024, 2051)

# x = Year
# t = x - 2024  ← explicit shifted time variable
T = YEARS - 2024

# ======================================================
# FIX TOTAL ROW — FILL 2024 VALUES
# ======================================================
mask_total = df["County"] == "TOTAL"

if mask_total.any():

    non_total = df.loc[~mask_total]

    # --- Superchargers & EVs: SUM ---
    df.loc[mask_total, "Superchargers_2024"] = non_total["Superchargers_2024"].sum()
    df.loc[mask_total, "EVs_2024"]           = non_total["EVs_2024"].sum()

    # --- Adoption: POPULATION-WEIGHTED ---
    weighted_adoption_2024 = (
        (non_total["Adoption_2024"] * non_total["Pop_2024"]).sum()
        / non_total["Pop_2024"].sum()
    )

    df.loc[mask_total, "Adoption_2024"] = weighted_adoption_2024

# ======================================================
# EXPORT FIXED DATA
# ======================================================
df.to_excel(output_path, index=False)
print("✅ TOTAL row fixed and exported:")
print(output_path)

# ======================================================
# PLOT EACH COUNTY + TOTAL
# Function plotted:
#   y = Superchargers(x)
#   x = Year
#   plotted using t = (x - 2024)
# ======================================================
for _, row in df.iterrows():

    county = row["County"]

    # y-values
    superchargers = np.array(
        [row[f"Superchargers_{y}"] for y in YEARS],
        dtype=float
    )

    # contextual covariate (NOT part of the function)
    adoption = np.array(
        [row[f"Adoption_{y}"] for y in YEARS],
        dtype=float
    )

    # --------------------------------------------------
    # FIGURE
    # --------------------------------------------------
    fig, ax = plt.subplots(figsize=(11, 7))

    # --------------------------------------------------
    # SCATTER
    # x-axis: t = (x - 2024)
    # y-axis: Superchargers
    # --------------------------------------------------
    ax.scatter(T, superchargers, s=35)
    ax.set_ylabel("Superchargers")
    ax.set_xlabel("x - 2024 (Time Index)")

    # --------------------------------------------------
    # BOTTOM X-AXIS — ADOPTION (LABEL ONLY)
    # --------------------------------------------------
    ax.set_xticks(T)
    ax.set_xticklabels(
        [f"{a*100:.0f}%" for a in adoption],
        rotation=90,
        fontsize=8
    )

    # --------------------------------------------------
    # TOP X-AXIS — YEAR (LABEL ONLY)
    # --------------------------------------------------
    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(T)
    ax_top.set_xticklabels(YEARS, rotation=90, fontsize=8)
    ax_top.set_xlabel("Year", labelpad=30)

    # --------------------------------------------------
    # TITLE
    # --------------------------------------------------
    ax.set_title(county, fontsize=12)

    # --------------------------------------------------
    # SAVE
    # --------------------------------------------------
    safe_name = county.replace(" ", "_").replace("/", "_")
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, f"{safe_name}.png"), dpi=300)
    plt.close()

print("✅ Scatter charts generated")
print("📁 Charts folder:", output_folder)


✅ TOTAL row fixed and exported:
/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc_TOTAL_FIXED.xlsx
✅ Scatter charts generated
📁 Charts folder: /Users/judycheng/Desktop/county_charts_scatter


In [54]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.metrics import r2_score
from scipy.optimize import curve_fit

# ======================================================
# PATHS
# ======================================================
input_path  = "/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc_TOTAL_FIXED.xlsx"

desktop = os.path.join(os.path.expanduser("~"), "Desktop")
output_folder = os.path.join(desktop, "county_charts_bestfit")
os.makedirs(output_folder, exist_ok=True)

output_excel = os.path.join(
    desktop,
    "wa_county_supercharger_bestfit_equations.xlsx"
)

# ======================================================
# LOAD DATA
# ======================================================
df = pd.read_excel(input_path)
df["County"] = df["County"].astype(str).str.strip()

YEARS = np.arange(2024, 2051)
T = YEARS - 2024  # 🔑 shifted x

# ======================================================
# MODEL DEFINITIONS (x = t = Year-2024)
# ======================================================
def linear(t, a, b):
    return a*t + b

def quadratic(t, a, b, c):
    return a*t**2 + b*t + c

def cubic(t, a, b, c, d):
    return a*t**3 + b*t**2 + c*t + d

def logistic(t, L, k, t0):
    return L / (1 + np.exp(-k*(t - t0)))

models = {
    "Linear":    (linear,    2),
    "Quadratic": (quadratic, 3),
    "Cubic":     (cubic,     4),
    "Logistic":  (logistic,  3),
}

# ======================================================
# OUTPUT STORAGE
# ======================================================
results = []

# ======================================================
# LOOP BY COUNTY
# ======================================================
for _, row in df.iterrows():

    county = row["County"]

    y = np.array(
        [row[f"Superchargers_{y}"] for y in YEARS],
        dtype=float
    )

    best_r2 = -np.inf
    best_name = None
    best_fn = None
    best_params = None
    best_yhat = None

    # --------------------------------------------------
    # FIT CANDIDATES
    # --------------------------------------------------
    for name, (fn, n_params) in models.items():
        try:
            p0 = np.ones(n_params)
            params, _ = curve_fit(fn, T, y, p0=p0, maxfev=20000)
            y_hat = fn(T, *params)
            r2 = r2_score(y, y_hat)

            if r2 > best_r2:
                best_r2 = r2
                best_name = name
                best_fn = fn
                best_params = params
                best_yhat = y_hat

        except Exception:
            continue

    # --------------------------------------------------
    # FORMAT EQUATION (x - 2024)
    # --------------------------------------------------
    if best_name == "Linear":
        a, b = best_params
        eq = f"y = {a:.6e}*(x-2024) + {b:.6e}"

    elif best_name == "Quadratic":
        a, b, c = best_params
        eq = f"y = {a:.6e}*(x-2024)^2 + {b:.6e}*(x-2024) + {c:.6e}"

    elif best_name == "Cubic":
        a, b, c, d = best_params
        eq = (
            f"y = {a:.6e}*(x-2024)^3 + "
            f"{b:.6e}*(x-2024)^2 + "
            f"{c:.6e}*(x-2024) + "
            f"{d:.6e}"
        )

    elif best_name == "Logistic":
        L, k, t0 = best_params
        eq = (
            f"y = {L:.2f} / (1 + exp(-{k:.4f}*((x-2024)-{t0:.2f})))"
        )

    # --------------------------------------------------
    # SAVE RESULTS
    # --------------------------------------------------
    results.append({
        "County": county,
        "Best_Model": best_name,
        "R2": best_r2,
        "Equation_(x-2024)": eq
    })

    # --------------------------------------------------
    # PLOT (AXIS UNCHANGED)
    # --------------------------------------------------
    fig, ax = plt.subplots(figsize=(11, 7))

    ax.scatter(T, y, s=35, label="Observed")
    ax.plot(T, best_yhat, linewidth=2.5, label=f"Best Fit: {best_name}")

    ax.set_ylabel("Superchargers")
    ax.set_xlabel("x - 2024 (Time Index)")

    # bottom x-axis labels (Adoption — unchanged)
    adoption = np.array(
        [row[f"Adoption_{y}"] for y in YEARS],
        dtype=float
    )

    ax.set_xticks(T)
    ax.set_xticklabels(
        [f"{a*100:.0f}%" for a in adoption],
        rotation=90,
        fontsize=8
    )

    # top axis: Year
    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(T)
    ax_top.set_xticklabels(YEARS, rotation=90, fontsize=8)
    ax_top.set_xlabel("Year", labelpad=30)

    ax.set_title(county)
    ax.legend()

    plt.tight_layout()
    safe = county.replace(" ", "_").replace("/", "_")
    plt.savefig(os.path.join(output_folder, f"{safe}.png"), dpi=300)
    plt.close()

# ======================================================
# EXPORT EQUATIONS
# ======================================================
pd.DataFrame(results).to_excel(output_excel, index=False)

print("✅ Best-fit curves completed")
print("📈 Charts:", output_folder)
print("📄 Equations:", output_excel)


/var/folders/6f/2tytnt59249dq7mgqy_lsckr0000gn/T/ipykernel_47791/1205681957.py:44: RuntimeWarning: overflow encountered in exp
  return L / (1 + np.exp(-k*(t - t0)))
/var/folders/6f/2tytnt59249dq7mgqy_lsckr0000gn/T/ipykernel_47791/1205681957.py:82: OptimizeWarning: Covariance of the parameters could not be estimated
  params, _ = curve_fit(fn, T, y, p0=p0, maxfev=20000)
/var/folders/6f/2tytnt59249dq7mgqy_lsckr0000gn/T/ipykernel_47791/1205681957.py:82: OptimizeWarning: Covariance of the parameters could not be estimated
  params, _ = curve_fit(fn, T, y, p0=p0, maxfev=20000)
/var/folders/6f/2tytnt59249dq7mgqy_lsckr0000gn/T/ipykernel_47791/1205681957.py:82: OptimizeWarning: Covariance of the parameters could not be estimated
  params, _ = curve_fit(fn, T, y, p0=p0, maxfev=20000)
/var/folders/6f/2tytnt59249dq7mgqy_lsckr0000gn/T/ipykernel_47791/1205681957.py:44: RuntimeWarning: overflow encountered in exp
  return L / (1 + np.exp(-k*(t - t0)))
/var/folders/6f/2tytnt59249dq7mgqy_lsckr0000gn/

✅ Best-fit curves completed
📈 Charts: /Users/judycheng/Desktop/county_charts_bestfit
📄 Equations: /Users/judycheng/Desktop/wa_county_supercharger_bestfit_equations.xlsx


In [55]:
import pandas as pd
import numpy as np
import os
from sklearn.metrics import r2_score
from scipy.optimize import curve_fit

# ======================================================
# PATHS
# ======================================================
input_path = "/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc_TOTAL_FIXED.xlsx"

desktop = os.path.join(os.path.expanduser("~"), "Desktop")
output_excel = os.path.join(
    desktop,
    "wa_county_adoption_vs_year_functions.xlsx"
)

# ======================================================
# LOAD DATA
# ======================================================
df = pd.read_excel(input_path)
df["County"] = df["County"].astype(str).str.strip()

YEARS = np.arange(2024, 2051)

# ======================================================
# MODEL DEFINITIONS
# x = Year
# ======================================================
def linear(x, a, b):
    return a*(x-2024) + b

def quadratic(x, a, b, c):
    return a*(x-2024)**2 + b*(x-2024) + c

def logistic(x, L, k, x0):
    return L / (1 + np.exp(-k*((x-2024) - x0)))

models = {
    "Linear":    (linear, 2),
    "Quadratic": (quadratic, 3),
    "Logistic":  (logistic, 3),
}

# ======================================================
# FIT ADOPTION FUNCTIONS
# ======================================================
results = []

for _, row in df.iterrows():

    county = row["County"]

    # y = Adoption rate
    y = np.array(
        [row[f"Adoption_{y}"] for y in YEARS],
        dtype=float
    )

    best_r2 = -np.inf
    best_name = None
    best_params = None

    # ----------------------------------------------
    # Try each candidate model
    # ----------------------------------------------
    for name, (fn, n_params) in models.items():
        try:
            p0 = np.ones(n_params)
            params, _ = curve_fit(fn, YEARS, y, p0=p0, maxfev=20000)
            y_hat = fn(YEARS, *params)
            r2 = r2_score(y, y_hat)

            if r2 > best_r2:
                best_r2 = r2
                best_name = name
                best_params = params

        except Exception:
            continue

    # ----------------------------------------------
    # Format equation explicitly in (x - 2024)
    # ----------------------------------------------
    if best_name == "Linear":
        a, b = best_params
        equation = f"y = {a:.6e}*(x-2024) + {b:.6e}"

    elif best_name == "Quadratic":
        a, b, c = best_params
        equation = (
            f"y = {a:.6e}*(x-2024)^2 + "
            f"{b:.6e}*(x-2024) + "
            f"{c:.6e}"
        )

    elif best_name == "Logistic":
        L, k, x0 = best_params
        equation = (
            f"y = {L:.4f} / "
            f"(1 + exp(-{k:.4f}*((x-2024)-{x0:.4f})))"
        )

    results.append({
        "County": county,
        "Best_Model": best_name,
        "R2": best_r2,
        "Adoption_Function_(x=Year)": equation
    })

# ======================================================
# EXPORT FUNCTIONS ONLY
# ======================================================
pd.DataFrame(results).to_excel(output_excel, index=False)

print("✅ Adoption vs Year functions generated")
print("📄 Output file:", output_excel)


✅ Adoption vs Year functions generated
📄 Output file: /Users/judycheng/Desktop/wa_county_adoption_vs_year_functions.xlsx


In [56]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.metrics import r2_score
from scipy.optimize import curve_fit

# ======================================================
# PATHS
# ======================================================
input_path  = "/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc_TOTAL_FIXED.xlsx"

desktop = os.path.join(os.path.expanduser("~"), "Desktop")
output_folder = os.path.join(desktop, "county_charts_bestfit")
os.makedirs(output_folder, exist_ok=True)

output_excel = os.path.join(
    desktop,
    "wa_county_supercharger_bestfit_equations.xlsx"
)

# ======================================================
# LOAD DATA
# ======================================================
df = pd.read_excel(input_path)
df["County"] = df["County"].astype(str).str.strip()

YEARS = np.arange(2024, 2051)
T = YEARS - 2024  # shifted x

# ======================================================
# MODEL DEFINITIONS
# ======================================================
def linear(t, a, b):
    return a*t + b

def quadratic(t, a, b, c):
    return a*t**2 + b*t + c

def cubic(t, a, b, c, d):
    return a*t**3 + b*t**2 + c*t + d

def logistic(t, L, k, t0):
    return L / (1 + np.exp(-k*(t - t0)))

models = {
    "Linear":    (linear,    2),
    "Quadratic": (quadratic, 3),
    "Cubic":     (cubic,     4),
    "Logistic":  (logistic,  3),
}

# ======================================================
# OUTPUT STORAGE
# ======================================================
results = []

# ======================================================
# LOOP BY COUNTY
# ======================================================
for _, row in df.iterrows():

    county = row["County"]

    # ----------------------------------------------
    # Superchargers (y1)
    # ----------------------------------------------
    y_sc = np.array(
        [row[f"Superchargers_{y}"] for y in YEARS],
        dtype=float
    )

    # ----------------------------------------------
    # Adoption (y2)
    # ----------------------------------------------
    adoption = np.array(
        [row[f"Adoption_{y}"] for y in YEARS],
        dtype=float
    )

    best_r2 = -np.inf
    best_name = None
    best_params = None
    best_yhat = None

    # ----------------------------------------------
    # FIT SUPERCHARGER MODELS
    # ----------------------------------------------
    for name, (fn, n_params) in models.items():
        try:
            p0 = np.ones(n_params)
            params, _ = curve_fit(fn, T, y_sc, p0=p0, maxfev=20000)
            y_hat = fn(T, *params)
            r2 = r2_score(y_sc, y_hat)

            if r2 > best_r2:
                best_r2 = r2
                best_name = name
                best_params = params
                best_yhat = y_hat

        except Exception:
            continue

    # ----------------------------------------------
    # FORMAT EQUATION
    # ----------------------------------------------
    if best_name == "Linear":
        a, b = best_params
        eq = f"y = {a:.6e}*(x-2024) + {b:.6e}"

    elif best_name == "Quadratic":
        a, b, c = best_params
        eq = f"y = {a:.6e}*(x-2024)^2 + {b:.6e}*(x-2024) + {c:.6e}"

    elif best_name == "Cubic":
        a, b, c, d = best_params
        eq = (
            f"y = {a:.6e}*(x-2024)^3 + "
            f"{b:.6e}*(x-2024)^2 + "
            f"{c:.6e}*(x-2024) + "
            f"{d:.6e}"
        )

    elif best_name == "Logistic":
        L, k, t0 = best_params
        eq = f"y = {L:.2f} / (1 + exp(-{k:.4f}*((x-2024)-{t0:.2f})))"

    results.append({
        "County": county,
        "Best_Model": best_name,
        "R2": best_r2,
        "Equation_(x-2024)": eq
    })

    # ==================================================
    # PLOT (AXES PRESERVED)
    # ==================================================
    fig, ax = plt.subplots(figsize=(11, 7))

    # ---- Superchargers (LEFT Y) ----
    ax.scatter(T, y_sc, s=35, label="Superchargers (Observed)")
    ax.plot(T, best_yhat, linewidth=2.5, label=f"SC Best Fit: {best_name}")
    ax.set_ylabel("Superchargers")

    # ---- Adoption (RIGHT Y) ----
    ax2 = ax.twinx()
    ax2.plot(T, adoption, linestyle="--", linewidth=2.0, label="Adoption Rate")
    ax2.set_ylabel("Adoption Rate")
    ax2.set_ylim(0, 1)   # adoption is a rate

    # ---- X AXIS (UNCHANGED) ----
    ax.set_xlabel("x - 2024 (Time Index)")
    ax.set_xticks(T)
    ax.set_xticklabels(
        [f"{a*100:.0f}%" for a in adoption],
        rotation=90,
        fontsize=8
    )

    # ---- TOP X AXIS (YEAR) ----
    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(T)
    ax_top.set_xticklabels(YEARS, rotation=90, fontsize=8)
    ax_top.set_xlabel("Year", labelpad=30)

    # ---- TITLE + LEGEND ----
    ax.set_title(county)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

    # ---- SAVE ----
    plt.tight_layout()
    safe = county.replace(" ", "_").replace("/", "_")
    plt.savefig(os.path.join(output_folder, f"{safe}.png"), dpi=300)
    plt.close()

# ======================================================
# EXPORT EQUATIONS
# ======================================================
pd.DataFrame(results).to_excel(output_excel, index=False)

print("✅ Superchargers + Adoption curves plotted")
print("📈 Charts:", output_folder)
print("📄 Equations:", output_excel)


/var/folders/6f/2tytnt59249dq7mgqy_lsckr0000gn/T/ipykernel_47791/1556502092.py:44: RuntimeWarning: overflow encountered in exp
  return L / (1 + np.exp(-k*(t - t0)))
/var/folders/6f/2tytnt59249dq7mgqy_lsckr0000gn/T/ipykernel_47791/1556502092.py:92: OptimizeWarning: Covariance of the parameters could not be estimated
  params, _ = curve_fit(fn, T, y_sc, p0=p0, maxfev=20000)
/var/folders/6f/2tytnt59249dq7mgqy_lsckr0000gn/T/ipykernel_47791/1556502092.py:92: OptimizeWarning: Covariance of the parameters could not be estimated
  params, _ = curve_fit(fn, T, y_sc, p0=p0, maxfev=20000)
/var/folders/6f/2tytnt59249dq7mgqy_lsckr0000gn/T/ipykernel_47791/1556502092.py:92: OptimizeWarning: Covariance of the parameters could not be estimated
  params, _ = curve_fit(fn, T, y_sc, p0=p0, maxfev=20000)
/var/folders/6f/2tytnt59249dq7mgqy_lsckr0000gn/T/ipykernel_47791/1556502092.py:44: RuntimeWarning: overflow encountered in exp
  return L / (1 + np.exp(-k*(t - t0)))
/var/folders/6f/2tytnt59249dq7mgqy_lsc

✅ Superchargers + Adoption curves plotted
📈 Charts: /Users/judycheng/Desktop/county_charts_bestfit
📄 Equations: /Users/judycheng/Desktop/wa_county_supercharger_bestfit_equations.xlsx


In [57]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.metrics import r2_score
from scipy.optimize import curve_fit

# ======================================================
# PATHS
# ======================================================
input_path  = "/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc_TOTAL_FIXED.xlsx"

desktop = os.path.join(os.path.expanduser("~"), "Desktop")
output_folder = os.path.join(desktop, "county_charts_bestfit")
os.makedirs(output_folder, exist_ok=True)

output_excel = os.path.join(
    desktop,
    "wa_county_supercharger_bestfit_equations.xlsx"
)

# ======================================================
# LOAD DATA
# ======================================================
df = pd.read_excel(input_path)
df["County"] = df["County"].astype(str).str.strip()

YEARS = np.arange(2024, 2051)
T = YEARS - 2024

# ======================================================
# MODEL DEFINITIONS
# ======================================================
def linear(t, a, b):
    return a*t + b

def quadratic(t, a, b, c):
    return a*t**2 + b*t + c

def cubic(t, a, b, c, d):
    return a*t**3 + b*t**2 + c*t + d

def logistic(t, L, k, t0):
    return L / (1 + np.exp(-k*(t - t0)))

models = {
    "Linear":    (linear,    2),
    "Quadratic": (quadratic, 3),
    "Cubic":     (cubic,     4),
    "Logistic":  (logistic,  3),
}

# ======================================================
# OUTPUT STORAGE
# ======================================================
results = []

# ======================================================
# LOOP BY COUNTY
# ======================================================
for _, row in df.iterrows():

    county = row["County"]

    y_sc = np.array(
        [row[f"Superchargers_{y}"] for y in YEARS],
        dtype=float
    )

    adoption = np.array(
        [row[f"Adoption_{y}"] for y in YEARS],
        dtype=float
    )

    best_r2 = -np.inf
    best_name = None
    best_params = None
    best_yhat = None

    for name, (fn, n_params) in models.items():
        try:
            params, _ = curve_fit(fn, T, y_sc, p0=np.ones(n_params), maxfev=20000)
            y_hat = fn(T, *params)
            r2 = r2_score(y_sc, y_hat)

            if r2 > best_r2:
                best_r2 = r2
                best_name = name
                best_params = params
                best_yhat = y_hat

        except Exception:
            continue

    if best_name == "Linear":
        a, b = best_params
        eq = f"y = {a:.6e}*(x-2024) + {b:.6e}"

    elif best_name == "Quadratic":
        a, b, c = best_params
        eq = f"y = {a:.6e}*(x-2024)^2 + {b:.6e}*(x-2024) + {c:.6e}"

    elif best_name == "Cubic":
        a, b, c, d = best_params
        eq = (
            f"y = {a:.6e}*(x-2024)^3 + "
            f"{b:.6e}*(x-2024)^2 + "
            f"{c:.6e}*(x-2024) + "
            f"{d:.6e}"
        )

    elif best_name == "Logistic":
        L, k, t0 = best_params
        eq = f"y = {L:.2f} / (1 + exp(-{k:.4f}*((x-2024)-{t0:.2f})))"

    results.append({
        "County": county,
        "Best_Model": best_name,
        "R2": best_r2,
        "Equation_(x-2024)": eq
    })

    # ==================================================
    # PLOT
    # ==================================================
    fig, ax = plt.subplots(figsize=(11, 7))

    ax.scatter(T, y_sc, s=35, label="Superchargers (Observed)")
    ax.plot(T, best_yhat, linewidth=2.5, label=f"SC Best Fit: {best_name}")
    ax.set_ylabel("Superchargers")

    ax2 = ax.twinx()
    ax2.plot(T, adoption, linestyle="--", linewidth=2.0, label="Adoption Rate")
    ax2.set_ylabel("Adoption Rate")
    ax2.set_ylim(0, 1)

    # ---- REMOVE BOTTOM X AXIS ----
    ax.set_xlabel("")
    ax.set_xticks([])
    ax.tick_params(bottom=False)

    # ---- TOP X AXIS (YEAR) ----
    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(T)
    ax_top.set_xticklabels(YEARS, rotation=90, fontsize=8)
    ax_top.set_xlabel("Year", labelpad=30)

    ax.set_title(county)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

    plt.tight_layout()
    safe = county.replace(" ", "_").replace("/", "_")
    plt.savefig(os.path.join(output_folder, f"{safe}.png"), dpi=300)
    plt.close()

# ======================================================
# EXPORT
# ======================================================
pd.DataFrame(results).to_excel(output_excel, index=False)

print("✅ Bottom x-axis removed; charts preserved")
print("📈 Charts:", output_folder)
print("📄 Equations:", output_excel)


/var/folders/6f/2tytnt59249dq7mgqy_lsckr0000gn/T/ipykernel_47791/1372629812.py:44: RuntimeWarning: overflow encountered in exp
  return L / (1 + np.exp(-k*(t - t0)))
/var/folders/6f/2tytnt59249dq7mgqy_lsckr0000gn/T/ipykernel_47791/1372629812.py:82: OptimizeWarning: Covariance of the parameters could not be estimated
  params, _ = curve_fit(fn, T, y_sc, p0=np.ones(n_params), maxfev=20000)
/var/folders/6f/2tytnt59249dq7mgqy_lsckr0000gn/T/ipykernel_47791/1372629812.py:82: OptimizeWarning: Covariance of the parameters could not be estimated
  params, _ = curve_fit(fn, T, y_sc, p0=np.ones(n_params), maxfev=20000)
/var/folders/6f/2tytnt59249dq7mgqy_lsckr0000gn/T/ipykernel_47791/1372629812.py:82: OptimizeWarning: Covariance of the parameters could not be estimated
  params, _ = curve_fit(fn, T, y_sc, p0=np.ones(n_params), maxfev=20000)
/var/folders/6f/2tytnt59249dq7mgqy_lsckr0000gn/T/ipykernel_47791/1372629812.py:44: RuntimeWarning: overflow encountered in exp
  return L / (1 + np.exp(-k*(t -

✅ Bottom x-axis removed; charts preserved
📈 Charts: /Users/judycheng/Desktop/county_charts_bestfit
📄 Equations: /Users/judycheng/Desktop/wa_county_supercharger_bestfit_equations.xlsx


In [58]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.metrics import r2_score
from scipy.optimize import curve_fit

# ======================================================
# PATHS
# ======================================================
input_path  = "/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc_TOTAL_FIXED.xlsx"

desktop = os.path.join(os.path.expanduser("~"), "Desktop")
output_folder = os.path.join(desktop, "county_charts_bestfit")
os.makedirs(output_folder, exist_ok=True)

output_excel = os.path.join(
    desktop,
    "wa_county_supercharger_bestfit_equations.xlsx"
)

# ======================================================
# LOAD DATA
# ======================================================
df = pd.read_excel(input_path)
df["County"] = df["County"].astype(str).str.strip()

YEARS = np.arange(2024, 2051)
T = YEARS - 2024

# ======================================================
# MODEL DEFINITIONS
# ======================================================
def linear(t, a, b):
    return a*t + b

def quadratic(t, a, b, c):
    return a*t**2 + b*t + c

def cubic(t, a, b, c, d):
    return a*t**3 + b*t**2 + c*t + d

def logistic(t, L, k, t0):
    return L / (1 + np.exp(-k*(t - t0)))

models = {
    "Linear":    (linear,    2),
    "Quadratic": (quadratic, 3),
    "Cubic":     (cubic,     4),
    "Logistic":  (logistic,  3),
}

# ======================================================
# OUTPUT STORAGE
# ======================================================
results = []

# ======================================================
# LOOP BY COUNTY
# ======================================================
for _, row in df.iterrows():

    county = row["County"]

    y_sc = np.array(
        [row[f"Superchargers_{y}"] for y in YEARS],
        dtype=float
    )

    adoption = np.array(
        [row[f"Adoption_{y}"] for y in YEARS],
        dtype=float
    )

    best_r2 = -np.inf
    best_name = None
    best_params = None
    best_yhat = None

    # ----------------------------------------------
    # FIT SUPERCHARGER MODELS
    # ----------------------------------------------
    for name, (fn, n_params) in models.items():
        try:
            params, _ = curve_fit(fn, T, y_sc, p0=np.ones(n_params), maxfev=20000)
            y_hat = fn(T, *params)
            r2 = r2_score(y_sc, y_hat)

            if r2 > best_r2:
                best_r2 = r2
                best_name = name
                best_params = params
                best_yhat = y_hat

        except Exception:
            continue

    # ----------------------------------------------
    # FORMAT EQUATION
    # ----------------------------------------------
    if best_name == "Linear":
        a, b = best_params
        eq = f"y = {a:.6e}*(x-2024) + {b:.6e}"

    elif best_name == "Quadratic":
        a, b, c = best_params
        eq = f"y = {a:.6e}*(x-2024)^2 + {b:.6e}*(x-2024) + {c:.6e}"

    elif best_name == "Cubic":
        a, b, c, d = best_params
        eq = (
            f"y = {a:.6e}*(x-2024)^3 + "
            f"{b:.6e}*(x-2024)^2 + "
            f"{c:.6e}*(x-2024) + "
            f"{d:.6e}"
        )

    elif best_name == "Logistic":
        L, k, t0 = best_params
        eq = f"y = {L:.2f} / (1 + exp(-{k:.4f}*((x-2024)-{t0:.2f})))"

    results.append({
        "County": county,
        "Best_Model": best_name,
        "R2": best_r2,
        "Equation_(x-2024)": eq
    })

    # ==================================================
    # PLOT
    # ==================================================
    fig, ax = plt.subplots(figsize=(11, 7))

    # ---- Superchargers: LINE ONLY (scatter removed) ----
    ax.plot(T, best_yhat, linewidth=2.5, label=f"Superchargers ({best_name})")
    ax.set_ylabel("Superchargers")

    # ---- Adoption (RIGHT Y) — RED ----
    ax2 = ax.twinx()
    ax2.plot(
        T,
        adoption,
        linestyle="--",
        linewidth=2.5,
        color="red",
        label="Adoption Rate"
    )

    ax2.set_ylabel("Adoption Rate", color="red")
    ax2.tick_params(axis="y", colors="red")
    ax2.set_ylim(0, 1)

    # Optional: make right spine red for clarity
    ax2.spines["right"].set_color("red")

    # ---- REMOVE BOTTOM X AXIS ----
    ax.set_xlabel("")
    ax.set_xticks([])
    ax.tick_params(bottom=False)

    # ---- TOP X AXIS (YEAR) ----
    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(T)
    ax_top.set_xticklabels(YEARS, rotation=90, fontsize=8)
    ax_top.set_xlabel("Year", labelpad=30)

    # ---- TITLE + LEGEND ----
    ax.set_title(county)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

    # ---- SAVE ----
    plt.tight_layout()
    safe = county.replace(" ", "_").replace("/", "_")
    plt.savefig(os.path.join(output_folder, f"{safe}.png"), dpi=300)
    plt.close()

# ======================================================
# EXPORT EQUATIONS
# ======================================================
pd.DataFrame(results).to_excel(output_excel, index=False)

print("✅ Scatter removed; adoption curve & axis styled red")
print("📈 Charts:", output_folder)
print("📄 Equations:", output_excel)


/var/folders/6f/2tytnt59249dq7mgqy_lsckr0000gn/T/ipykernel_47791/1885059940.py:44: RuntimeWarning: overflow encountered in exp
  return L / (1 + np.exp(-k*(t - t0)))
/var/folders/6f/2tytnt59249dq7mgqy_lsckr0000gn/T/ipykernel_47791/1885059940.py:85: OptimizeWarning: Covariance of the parameters could not be estimated
  params, _ = curve_fit(fn, T, y_sc, p0=np.ones(n_params), maxfev=20000)
/var/folders/6f/2tytnt59249dq7mgqy_lsckr0000gn/T/ipykernel_47791/1885059940.py:85: OptimizeWarning: Covariance of the parameters could not be estimated
  params, _ = curve_fit(fn, T, y_sc, p0=np.ones(n_params), maxfev=20000)
/var/folders/6f/2tytnt59249dq7mgqy_lsckr0000gn/T/ipykernel_47791/1885059940.py:85: OptimizeWarning: Covariance of the parameters could not be estimated
  params, _ = curve_fit(fn, T, y_sc, p0=np.ones(n_params), maxfev=20000)
/var/folders/6f/2tytnt59249dq7mgqy_lsckr0000gn/T/ipykernel_47791/1885059940.py:44: RuntimeWarning: overflow encountered in exp
  return L / (1 + np.exp(-k*(t -

✅ Scatter removed; adoption curve & axis styled red
📈 Charts: /Users/judycheng/Desktop/county_charts_bestfit
📄 Equations: /Users/judycheng/Desktop/wa_county_supercharger_bestfit_equations.xlsx


In [59]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.metrics import r2_score
from scipy.optimize import curve_fit

# ======================================================
# PATHS
# ======================================================
input_path = "/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc_TOTAL_FIXED.xlsx"

desktop = os.path.join(os.path.expanduser("~"), "Desktop")
output_folder = os.path.join(desktop, "county_charts_baseline_anchored")
os.makedirs(output_folder, exist_ok=True)

output_excel = os.path.join(
    desktop,
    "wa_county_baseline_anchored_equations.xlsx"
)

# ======================================================
# LOAD DATA
# ======================================================
df = pd.read_excel(input_path)
df["County"] = df["County"].astype(str).str.strip()

YEARS = np.arange(2024, 2051)
T = YEARS - 2024   # shifted time index
T_FIT = T[1:]      # exclude t=0 from fitting (baseline fixed)

# ======================================================
# BASELINE-ANCHORED MODELS
# y(0) = y0 EXACT
# ======================================================
def lin_anchor(t, a, y0):
    return a*t + y0

def quad_anchor(t, a, b, y0):
    return a*t**2 + b*t + y0

def cubic_anchor(t, a, b, c, y0):
    return a*t**3 + b*t**2 + c*t + y0

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def logistic_anchor(t, k, t0, y0, yT, tT=26.0):
    """
    Anchored logistic:
    y(0) = y0
    y(tT) = yT
    """
    s0 = sigmoid(k*(0 - t0))
    sT = sigmoid(k*(tT - t0))
    L = (yT - y0) / (sT - s0)
    return y0 + L * (sigmoid(k*(t - t0)) - s0)

# ======================================================
# MODEL CANDIDATES
# ======================================================
SC_MODELS = {
    "Linear":    ("lin",    1),
    "Quadratic": ("quad",   2),
    "Cubic":     ("cubic",  3),
    "Logistic":  ("logistic", 2),
}

AD_MODELS = {
    "Linear":    ("lin",    1),
    "Quadratic": ("quad",   2),
    "Logistic":  ("logistic", 2),
}

# ======================================================
# OUTPUT STORAGE
# ======================================================
results = []

# ======================================================
# LOOP BY COUNTY
# ======================================================
for _, row in df.iterrows():

    county = row["County"]

    # -----------------------------
    # DATA
    # -----------------------------
    sc = np.array([row[f"Superchargers_{y}"] for y in YEARS], float)
    ad = np.array([row[f"Adoption_{y}"]       for y in YEARS], float)

    sc0, scT = sc[0], sc[-1]
    ad0, adT = ad[0], ad[-1]

    # ==================================================
    # FIT SUPERCHARGERS (BASELINE-ANCHORED)
    # ==================================================
    best_sc = {"r2": -np.inf}

    for name, (kind, n_params) in SC_MODELS.items():
        try:
            if kind == "lin":
                fn = lambda t, a: lin_anchor(t, a, sc0)
                params, _ = curve_fit(fn, T_FIT, sc[1:])
                yhat = fn(T, *params)

            elif kind == "quad":
                fn = lambda t, a, b: quad_anchor(t, a, b, sc0)
                params, _ = curve_fit(fn, T_FIT, sc[1:])
                yhat = fn(T, *params)

            elif kind == "cubic":
                fn = lambda t, a, b, c: cubic_anchor(t, a, b, c, sc0)
                params, _ = curve_fit(fn, T_FIT, sc[1:])
                yhat = fn(T, *params)

            elif kind == "logistic":
                fn = lambda t, k, t0: logistic_anchor(t, k, t0, sc0, scT)
                params, _ = curve_fit(fn, T_FIT, sc[1:])
                yhat = fn(T, *params)

            r2 = r2_score(sc, yhat)

            if r2 > best_sc["r2"]:
                best_sc = dict(name=name, params=params, yhat=yhat, r2=r2)

        except Exception:
            continue

    # ==================================================
    # FIT ADOPTION (BASELINE-ANCHORED)
    # ==================================================
    best_ad = {"r2": -np.inf}

    for name, (kind, n_params) in AD_MODELS.items():
        try:
            if kind == "lin":
                fn = lambda t, a: lin_anchor(t, a, ad0)
                params, _ = curve_fit(fn, T_FIT, ad[1:])
                yhat = fn(T, *params)

            elif kind == "quad":
                fn = lambda t, a, b: quad_anchor(t, a, b, ad0)
                params, _ = curve_fit(fn, T_FIT, ad[1:])
                yhat = fn(T, *params)

            elif kind == "logistic":
                fn = lambda t, k, t0: logistic_anchor(t, k, t0, ad0, adT)
                params, _ = curve_fit(fn, T_FIT, ad[1:])
                yhat = fn(T, *params)

            r2 = r2_score(ad, yhat)

            if r2 > best_ad["r2"]:
                best_ad = dict(name=name, params=params, yhat=yhat, r2=r2)

        except Exception:
            continue

    # ==================================================
    # STORE RESULTS
    # ==================================================
    results.append({
        "County": county,
        "SC_Model": best_sc["name"],
        "SC_R2": best_sc["r2"],
        "Adoption_Model": best_ad["name"],
        "Adoption_R2": best_ad["r2"],
    })

    # ==================================================
    # PLOT (STYLE PRESERVED)
    # ==================================================
    fig, ax = plt.subplots(figsize=(11, 7))

    # Superchargers — line only
    ax.plot(T, best_sc["yhat"], linewidth=2.5, label="Superchargers")
    ax.set_ylabel("Superchargers")

    # Adoption — red axis
    ax2 = ax.twinx()
    ax2.plot(T, best_ad["yhat"], "--", linewidth=2.5, color="red", label="Adoption")
    ax2.set_ylabel("Adoption Rate", color="red")
    ax2.tick_params(axis="y", colors="red")
    ax2.set_ylim(0, 1)
    ax2.spines["right"].set_color("red")

    # Remove bottom x-axis
    ax.set_xticks([])
    ax.tick_params(bottom=False)

    # Top x-axis (Year)
    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(T)
    ax_top.set_xticklabels(YEARS, rotation=90, fontsize=8)
    ax_top.set_xlabel("Year", labelpad=30)

    ax.set_title(county)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

    plt.tight_layout()
    safe = county.replace(" ", "_")
    plt.savefig(os.path.join(output_folder, f"{safe}.png"), dpi=300)
    plt.close()

# ======================================================
# EXPORT SUMMARY
# ======================================================
pd.DataFrame(results).to_excel(output_excel, index=False)

print("✅ 2024 baseline enforced for Adoption & Superchargers")
print("📈 Charts:", output_folder)
print("📄 Summary:", output_excel)


✅ 2024 baseline enforced for Adoption & Superchargers
📈 Charts: /Users/judycheng/Desktop/county_charts_baseline_anchored
📄 Summary: /Users/judycheng/Desktop/wa_county_baseline_anchored_equations.xlsx


In [60]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.metrics import r2_score
from scipy.optimize import curve_fit

# ======================================================
# PATHS
# ======================================================
input_path = "/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc_TOTAL_FIXED.xlsx"

desktop = os.path.join(os.path.expanduser("~"), "Desktop")
output_folder = os.path.join(desktop, "county_charts_baseline_anchored")
os.makedirs(output_folder, exist_ok=True)

output_excel = os.path.join(
    desktop,
    "wa_county_baseline_anchored_equations.xlsx"
)

# ======================================================
# LOAD DATA
# ======================================================
df = pd.read_excel(input_path)
df["County"] = df["County"].astype(str).str.strip()

YEARS = np.arange(2024, 2051)
T = YEARS - 2024
T_FIT = T[1:]   # exclude baseline year

# ======================================================
# BASELINE-ANCHORED FUNCTIONS
# ======================================================
def lin_anchor(t, a, y0):
    return a*t + y0

def quad_anchor(t, a, b, y0):
    return a*t**2 + b*t + y0

def cubic_anchor(t, a, b, c, y0):
    return a*t**3 + b*t**2 + c*t + y0

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def logistic_anchor(t, k, t0, y0, yT, tT=26):
    s0 = sigmoid(k*(0 - t0))
    sT = sigmoid(k*(tT - t0))
    L = (yT - y0) / (sT - s0)
    return y0 + L * (sigmoid(k*(t - t0)) - s0)

# ======================================================
# MODEL CANDIDATES
# ======================================================
SC_MODELS = ["Linear", "Quadratic", "Cubic", "Logistic"]
AD_MODELS = ["Linear", "Quadratic", "Logistic"]

# ======================================================
# OUTPUT STORAGE
# ======================================================
rows_out = []

# ======================================================
# LOOP BY COUNTY
# ======================================================
for _, row in df.iterrows():

    county = row["County"]

    sc = np.array([row[f"Superchargers_{y}"] for y in YEARS], float)
    ad = np.array([row[f"Adoption_{y}"]       for y in YEARS], float)

    sc0, scT = sc[0], sc[-1]
    ad0, adT = ad[0], ad[-1]

    # ==================================================
    # FIT SUPERCHARGERS
    # ==================================================
    best_sc = {"r2": -np.inf}

    for name in SC_MODELS:
        try:
            if name == "Linear":
                fn = lambda t, a: lin_anchor(t, a, sc0)
                params, _ = curve_fit(fn, T_FIT, sc[1:])
                yhat = fn(T, *params)
                eq_sc = f"y = {params[0]:.6e}*(x-2024) + {sc0:.6e}"

            elif name == "Quadratic":
                fn = lambda t, a, b: quad_anchor(t, a, b, sc0)
                params, _ = curve_fit(fn, T_FIT, sc[1:])
                yhat = fn(T, *params)
                eq_sc = (
                    f"y = {params[0]:.6e}*(x-2024)^2 + "
                    f"{params[1]:.6e}*(x-2024) + "
                    f"{sc0:.6e}"
                )

            elif name == "Cubic":
                fn = lambda t, a, b, c: cubic_anchor(t, a, b, c, sc0)
                params, _ = curve_fit(fn, T_FIT, sc[1:])
                yhat = fn(T, *params)
                eq_sc = (
                    f"y = {params[0]:.6e}*(x-2024)^3 + "
                    f"{params[1]:.6e}*(x-2024)^2 + "
                    f"{params[2]:.6e}*(x-2024) + "
                    f"{sc0:.6e}"
                )

            elif name == "Logistic":
                fn = lambda t, k, t0: logistic_anchor(t, k, t0, sc0, scT)
                params, _ = curve_fit(fn, T_FIT, sc[1:])
                yhat = fn(T, *params)
                eq_sc = (
                    f"y = {sc0:.2f} + L*(sigmoid({params[0]:.4f}"
                    f"*((x-2024)-{params[1]:.2f})) - sigmoid(-{params[0]:.4f}*{params[1]:.2f}))"
                )

            r2 = r2_score(sc, yhat)
            if r2 > best_sc["r2"]:
                best_sc = dict(model=name, r2=r2, eq=eq_sc)

        except Exception:
            continue

    # ==================================================
    # FIT ADOPTION
    # ==================================================
    best_ad = {"r2": -np.inf}

    for name in AD_MODELS:
        try:
            if name == "Linear":
                fn = lambda t, a: lin_anchor(t, a, ad0)
                params, _ = curve_fit(fn, T_FIT, ad[1:])
                yhat = fn(T, *params)
                eq_ad = f"y = {params[0]:.6e}*(x-2024) + {ad0:.6e}"

            elif name == "Quadratic":
                fn = lambda t, a, b: quad_anchor(t, a, b, ad0)
                params, _ = curve_fit(fn, T_FIT, ad[1:])
                yhat = fn(T, *params)
                eq_ad = (
                    f"y = {params[0]:.6e}*(x-2024)^2 + "
                    f"{params[1]:.6e}*(x-2024) + "
                    f"{ad0:.6e}"
                )

            elif name == "Logistic":
                fn = lambda t, k, t0: logistic_anchor(t, k, t0, ad0, adT)
                params, _ = curve_fit(fn, T_FIT, ad[1:])
                yhat = fn(T, *params)
                eq_ad = (
                    f"y = {ad0:.4f} + L*(sigmoid({params[0]:.4f}"
                    f"*((x-2024)-{params[1]:.2f})) - sigmoid(-{params[0]:.4f}*{params[1]:.2f}))"
                )

            r2 = r2_score(ad, yhat)
            if r2 > best_ad["r2"]:
                best_ad = dict(model=name, r2=r2, eq=eq_ad)

        except Exception:
            continue

    # ==================================================
    # STORE OUTPUT
    # ==================================================
    rows_out.append({
        "County": county,
        "Supercharger_Model": best_sc["model"],
        "Supercharger_R2": best_sc["r2"],
        "Supercharger_Equation_(x-2024)": best_sc["eq"],
        "Adoption_Model": best_ad["model"],
        "Adoption_R2": best_ad["r2"],
        "Adoption_Equation_(x-2024)": best_ad["eq"],
    })

# ======================================================
# EXPORT
# ======================================================
pd.DataFrame(rows_out).to_excel(output_excel, index=False)

print("✅ Both equations exported by county")
print("📄 Output:", output_excel)


✅ Both equations exported by county
📄 Output: /Users/judycheng/Desktop/wa_county_baseline_anchored_equations.xlsx


In [62]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.metrics import r2_score
from scipy.optimize import curve_fit

# ======================================================
# PATHS
# ======================================================
input_path = "/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc_TOTAL_FIXED.xlsx"

desktop = os.path.join(os.path.expanduser("~"), "Desktop")
output_folder = os.path.join(desktop, "county_charts_baseline_anchored")
os.makedirs(output_folder, exist_ok=True)

output_excel = os.path.join(
    desktop,
    "wa_county_baseline_anchored_equations.xlsx"
)

# ======================================================
# LOAD DATA
# ======================================================
df = pd.read_excel(input_path)
df["County"] = df["County"].astype(str).str.strip()

YEARS = np.arange(2024, 2051)
T = YEARS - 2024
T_FIT = T[1:]   # exclude baseline year

# ======================================================
# BASELINE-ANCHORED FUNCTIONS
# ======================================================
def lin_anchor(t, a, y0):
    return a*t + y0

def quad_anchor(t, a, b, y0):
    return a*t**2 + b*t + y0

def cubic_anchor(t, a, b, c, y0):
    return a*t**3 + b*t**2 + c*t + y0

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def logistic_anchor(t, k, t0, y0, yT, tT=26):
    s0 = sigmoid(k*(0 - t0))
    sT = sigmoid(k*(tT - t0))
    L = (yT - y0) / (sT - s0)
    return y0 + L * (sigmoid(k*(t - t0)) - s0)

# ======================================================
# MODEL CANDIDATES
# ======================================================
SC_MODELS = ["Linear", "Quadratic", "Cubic", "Logistic"]
AD_MODELS = ["Linear", "Quadratic", "Logistic"]

# ======================================================
# OUTPUT STORAGE
# ======================================================
results = []

# ======================================================
# LOOP BY COUNTY
# ======================================================
for _, row in df.iterrows():

    county = row["County"]

    sc = np.array([row[f"Superchargers_{y}"] for y in YEARS], float)
    ad = np.array([row[f"Adoption_{y}"] for y in YEARS], float)

    sc0, scT = sc[0], sc[-1]
    ad0, adT = ad[0], ad[-1]

    # ==================================================
    # FIT SUPERCHARGERS
    # ==================================================
    best_sc = {"r2": -np.inf}

    for name in SC_MODELS:
        try:
            if name == "Linear":
                fn = lambda t, a: lin_anchor(t, a, sc0)
                params, _ = curve_fit(fn, T_FIT, sc[1:])
                yhat = fn(T, *params)

            elif name == "Quadratic":
                fn = lambda t, a, b: quad_anchor(t, a, b, sc0)
                params, _ = curve_fit(fn, T_FIT, sc[1:])
                yhat = fn(T, *params)

            elif name == "Cubic":
                fn = lambda t, a, b, c: cubic_anchor(t, a, b, c, sc0)
                params, _ = curve_fit(fn, T_FIT, sc[1:])
                yhat = fn(T, *params)

            elif name == "Logistic":
                fn = lambda t, k, t0: logistic_anchor(t, k, t0, sc0, scT)
                params, _ = curve_fit(fn, T_FIT, sc[1:])
                yhat = fn(T, *params)

            r2 = r2_score(sc, yhat)
            if r2 > best_sc["r2"]:
                best_sc = dict(model=name, r2=r2, yhat=yhat)

        except Exception:
            continue

    # ==================================================
    # FIT ADOPTION
    # ==================================================
    best_ad = {"r2": -np.inf}

    for name in AD_MODELS:
        try:
            if name == "Linear":
                fn = lambda t, a: lin_anchor(t, a, ad0)
                params, _ = curve_fit(fn, T_FIT, ad[1:])
                yhat = fn(T, *params)

            elif name == "Quadratic":
                fn = lambda t, a, b: quad_anchor(t, a, b, ad0)
                params, _ = curve_fit(fn, T_FIT, ad[1:])
                yhat = fn(T, *params)

            elif name == "Logistic":
                fn = lambda t, k, t0: logistic_anchor(t, k, t0, ad0, adT)
                params, _ = curve_fit(fn, T_FIT, ad[1:])
                yhat = fn(T, *params)

            r2 = r2_score(ad, yhat)
            if r2 > best_ad["r2"]:
                best_ad = dict(model=name, r2=r2, yhat=yhat)

        except Exception:
            continue

    # ==================================================
    # STORE SUMMARY
    # ==================================================
    results.append({
        "County": county,
        "Supercharger_Model": best_sc["model"],
        "Supercharger_R2": best_sc["r2"],
        "Adoption_Model": best_ad["model"],
        "Adoption_R2": best_ad["r2"],
    })

    # ==================================================
    # PLOT (YEAR ON BOTTOM)
    # ==================================================
    fig, ax = plt.subplots(figsize=(11, 7))

    # Superchargers
    ax.plot(T, best_sc["yhat"], linewidth=2.5, label="Superchargers")
    ax.set_ylabel("Superchargers")

    # Adoption (red axis)
    ax2 = ax.twinx()
    ax2.plot(T, best_ad["yhat"], "--", linewidth=2.5, color="red", label="Adoption")
    ax2.set_ylabel("Adoption Rate", color="red")
    ax2.tick_params(axis="y", colors="red")
    ax2.set_ylim(0, 1)
    ax2.spines["right"].set_color("red")

    # Bottom X-axis = Year
    ax.set_xticks(T)
    ax.set_xticklabels(YEARS, rotation=90, fontsize=8)
    ax.set_xlabel("Year")

    ax.set_title(county)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

    plt.tight_layout()
    safe = county.replace(" ", "_")
    plt.savefig(os.path.join(output_folder, f"{safe}.png"), dpi=300)
    plt.close()

# ======================================================
# EXPORT SUMMARY
# ======================================================
pd.DataFrame(results).to_excel(output_excel, index=False)

print("✅ Charts updated: Year moved to bottom x-axis")
print("📈 Charts:", output_folder)
print("📄 Summary:", output_excel)


✅ Charts updated: Year moved to bottom x-axis
📈 Charts: /Users/judycheng/Desktop/county_charts_baseline_anchored
📄 Summary: /Users/judycheng/Desktop/wa_county_baseline_anchored_equations.xlsx


In [64]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.metrics import r2_score
from scipy.optimize import curve_fit

# ======================================================
# PATHS
# ======================================================
input_path = "/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc_TOTAL_FIXED.xlsx"

desktop = os.path.join(os.path.expanduser("~"), "Desktop")
output_folder = os.path.join(desktop, "county_charts_baseline_anchored")
os.makedirs(output_folder, exist_ok=True)

output_excel = os.path.join(
    desktop,
    "wa_county_baseline_anchored_equations.xlsx"
)

# ======================================================
# LOAD DATA
# ======================================================
df = pd.read_excel(input_path)
df["County"] = df["County"].astype(str).str.strip()

YEARS = np.arange(2024, 2051)
T = YEARS - 2024
T_FIT = T[1:]

# ======================================================
# BASELINE-ANCHORED FUNCTIONS
# ======================================================
def lin_anchor(t, a, y0):
    return a*t + y0

def quad_anchor(t, a, b, y0):
    return a*t**2 + b*t + y0

def cubic_anchor(t, a, b, c, y0):
    return a*t**3 + b*t**2 + c*t + y0

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def logistic_anchor(t, k, t0, y0, yT, tT=26):
    s0 = sigmoid(k*(0 - t0))
    sT = sigmoid(k*(tT - t0))
    L = (yT - y0) / (sT - s0)
    return y0 + L * (sigmoid(k*(t - t0)) - s0)

# ======================================================
# MODEL CANDIDATES
# ======================================================
SC_MODELS = ["Linear", "Quadratic", "Cubic", "Logistic"]
AD_MODELS = ["Linear", "Quadratic", "Logistic"]

# ======================================================
# OUTPUT STORAGE
# ======================================================
results = []

# ======================================================
# LOOP BY COUNTY
# ======================================================
for _, row in df.iterrows():

    county_raw = row["County"]
    county_name = "Statewide" if county_raw.upper() == "TOTAL" else county_raw

    sc = np.array([row[f"Superchargers_{y}"] for y in YEARS], float)
    ad = np.array([row[f"Adoption_{y}"] for y in YEARS], float)

    sc0, scT = sc[0], sc[-1]
    ad0, adT = ad[0], ad[-1]

    # ==================================================
    # FIT SUPERCHARGERS
    # ==================================================
    best_sc = {"r2": -np.inf}

    for name in SC_MODELS:
        try:
            if name == "Linear":
                fn = lambda t, a: lin_anchor(t, a, sc0)
                params, _ = curve_fit(fn, T_FIT, sc[1:])
                yhat = fn(T, *params)

            elif name == "Quadratic":
                fn = lambda t, a, b: quad_anchor(t, a, b, sc0)
                params, _ = curve_fit(fn, T_FIT, sc[1:])
                yhat = fn(T, *params)

            elif name == "Cubic":
                fn = lambda t, a, b, c: cubic_anchor(t, a, b, c, sc0)
                params, _ = curve_fit(fn, T_FIT, sc[1:])
                yhat = fn(T, *params)

            elif name == "Logistic":
                fn = lambda t, k, t0: logistic_anchor(t, k, t0, sc0, scT)
                params, _ = curve_fit(fn, T_FIT, sc[1:])
                yhat = fn(T, *params)

            r2 = r2_score(sc, yhat)
            if r2 > best_sc["r2"]:
                best_sc = dict(model=name, r2=r2, yhat=yhat)

        except Exception:
            continue

    # ==================================================
    # FIT ADOPTION
    # ==================================================
    best_ad = {"r2": -np.inf}

    for name in AD_MODELS:
        try:
            if name == "Linear":
                fn = lambda t, a: lin_anchor(t, a, ad0)
                params, _ = curve_fit(fn, T_FIT, ad[1:])
                yhat = fn(T, *params)

            elif name == "Quadratic":
                fn = lambda t, a, b: quad_anchor(t, a, b, ad0)
                params, _ = curve_fit(fn, T_FIT, ad[1:])
                yhat = fn(T, *params)

            elif name == "Logistic":
                fn = lambda t, k, t0: logistic_anchor(t, k, t0, ad0, adT)
                params, _ = curve_fit(fn, T_FIT, ad[1:])
                yhat = fn(T, *params)

            r2 = r2_score(ad, yhat)
            if r2 > best_ad["r2"]:
                best_ad = dict(model=name, r2=r2, yhat=yhat)

        except Exception:
            continue

    # ==================================================
    # STORE SUMMARY
    # ==================================================
    results.append({
        "County": county_name,
        "Supercharger_Model": best_sc["model"],
        "Supercharger_R2": best_sc["r2"],
        "Adoption_Model": best_ad["model"],
        "Adoption_R2": best_ad["r2"],
    })

    # ==================================================
    # PLOT
    # ==================================================
    fig, ax = plt.subplots(figsize=(11, 7))

    # Charge Points (BLUE)
    ax.plot(T, best_sc["yhat"], linewidth=2.5, color="blue", label="Charge Points")
    ax.set_ylabel("Total Charger Points", color="blue")
    ax.tick_params(axis="y", colors="blue")
    ax.spines["left"].set_color("blue")

    # Adoption (RED)
    ax2 = ax.twinx()
    ax2.plot(T, best_ad["yhat"], "--", linewidth=2.5, color="red", label="EV Adoption Rate")
    ax2.set_ylabel("EV Adoption Rate", color="red")
    ax2.tick_params(axis="y", colors="red")
    ax2.set_ylim(0, 1)
    ax2.spines["right"].set_color("red")

    # Bottom X-axis = Year
    ax.set_xticks(T)
    ax.set_xticklabels(YEARS, rotation=90, fontsize=8)
    ax.set_xlabel("Year")

    # ---- Dynamic Title ----
    ax.set_title(
        f"{county_name} – Time Based Forecast - Charger Built Out VS EV Adoption Rate",
        fontsize=13,
        pad=12
    )

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

    plt.tight_layout()
    safe = county_name.replace(" ", "_")
    plt.savefig(os.path.join(output_folder, f"{safe}.png"), dpi=300)
    plt.close()

# ======================================================
# EXPORT SUMMARY
# ======================================================
pd.DataFrame(results).to_excel(output_excel, index=False)

print("✅ Legend and titles updated")
print("📈 Charts:", output_folder)
print("📄 Summary:", output_excel)


✅ Legend and titles updated
📈 Charts: /Users/judycheng/Desktop/county_charts_baseline_anchored
📄 Summary: /Users/judycheng/Desktop/wa_county_baseline_anchored_equations.xlsx
